# RQ-VAE Recommender System

"Recommender Systems with Generative Retrieval"
Paper: https://arxiv.org/abs/2305.05065

In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml
import torch
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

from tecd_retail_recsys.data import DataPreprocessor
from tecd_retail_recsys.models.rqvae_recommender import RQVAERecommender
from tecd_retail_recsys.metrics import calculate_metrics

print(f"System version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"Numpy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available() if hasattr(torch.backends, 'mps') else False}")

System version: 3.11.14 (main, Oct 28 2025, 12:11:54) [Clang 20.1.4 ]
Pandas version: 2.3.3
Numpy version: 1.26.4
PyTorch version: 2.10.0
CUDA available: False
MPS available: True


## 1. Load Configuration

In [ ]:
with open('configs/rqvae.yaml', 'r') as f:
    config_raw = yaml.safe_load(f)

active = config_raw.get('active_config', 'B')
active_section = config_raw.get(f'config_{active}')
if active_section is None:
    raise ValueError(f"Config section 'config_{active}' not found in rqvae.yaml")

config = {
    'model': active_section['model'],
    'train': active_section['train'],
    'data': config_raw['data'],
    'evaluation': config_raw['evaluation'],
    'info': config_raw['info'],
}

print(f"Active config: {active}")
print(yaml.dump(config, default_flow_style=False))

Active config: C
data:
  day_begin: 1082
  day_end: 1308
  min_item_interactions: 20
  min_user_interactions: 5
  processed_data_dir: processed_data
  raw_data_path: t_ecd_small_partial/dataset/small
  test_days: 20
  users_limit: null
  val_days: 20
evaluation:
  k_values:
  - 10
  - 100
  metrics:
  - recall
  - ndcg
  - precision
  - map
  - mrr
info:
  checkpoint_name: rqvae_model.pt
  model_dir: ./models/rqvae/
  save_model: true
model:
  decoder:
    d_model: 256
    dim_feedforward: 1024
    dropout: 0.1
    max_seq_length: 150
    nhead: 8
    num_layers: 4
  item_embedding_dim: 300
  rqvae:
    codebook_size: 256
    commitment_cost: 0.25
    ema_decay: 0.995
    hidden_dims:
    - 512
    - 256
    num_quantizers: 4
train:
  decoder:
    batch_size: 128
    epochs: 10
    learning_rate: 0.0002
    warmup_epochs: 8
  rqvae:
    batch_size: 256
    epochs: 60
    learning_rate: 0.0003
  temperature: 1.0
  top_k: 100



## 2. Data Preprocessing

In [ ]:
preprocessor = DataPreprocessor(
    raw_data_path=config['data']['raw_data_path'],
    processed_data_dir=config['data']['processed_data_dir'],
    day_begin=config['data']['day_begin'],
    day_end=config['data']['day_end'],
    min_user_interactions=config['data']['min_user_interactions'],
    min_item_interactions=config['data']['min_item_interactions'],
    val_days=config['data']['val_days'],
    test_days=config['data']['test_days'],
    users_limit=config['data']['users_limit']
)

train_data, val_data, test_data = preprocessor.preprocess()

print(f"\nData splits:")
print(f"  Train: {len(train_data):,} events, {train_data['user_id'].nunique()} users, {train_data['item_id'].nunique()} items")
print(f"  Val:   {len(val_data):,} events, {val_data['user_id'].nunique()} users, {val_data['item_id'].nunique()} items")
print(f"  Test:  {len(test_data):,} events, {test_data['user_id'].nunique()} users, {test_data['item_id'].nunique()} items")

Starting data preprocessing...
Loading events from t_ecd_small_partial/dataset/small/retail/events
Loaded 236,479,226 total events
Loading items data from t_ecd_small_partial/dataset/small/retail/items.pq
Loaded 250,171 items with features: ['item_id', 'item_brand_id', 'item_category', 'item_subcategory', 'item_price', 'item_embedding']
Merged item features. Data shape: (236479226, 12)
Filtered to 3,758,762 events with action_type='added-to-cart'
After filtering (min_user_interactions=5, min_item_interactions=20): 3,189,167 events, 59,513 users, 30,546 items
Created mappings: 59513 users, 30546 items
Temporal split - Train: days < 1269 (900,527 events), Val: days 1269-1288 (227,843 events), Test: days >= 1289 (222,882 events)
Users in each part (train, val, test) - 7415

Data splits:
  Train: 900,527 events, 7415 users, 30354 items
  Val:   227,843 events, 7415 users, 26474 items
  Test:  222,882 events, 7415 users, 25825 items


## 3. Prepare Item Embeddings

In [ ]:
train_items = train_data[['item_id', 'item_embedding']].drop_duplicates('item_id')

print(f"Number of unique items in training: {len(train_items)}")

item_ids = train_items['item_id'].values
item_embeddings = np.stack(train_items['item_embedding'].values)

print(f"Item IDs range: {item_ids.min()} - {item_ids.max()}")
print(f"Item embeddings shape: {item_embeddings.shape}")

item_embeddings_tensor = torch.FloatTensor(item_embeddings)
item_ids_array = np.array(item_ids)

Number of unique items in training: 30354
Item IDs range: 0 - 30545
Item embeddings shape: (30354, 300)


## 4. Prepare User Sequences

In [ ]:
train_grouped = preprocessor.group_by_users(train_data, col_name='train_interactions')
val_grouped = preprocessor.group_by_users(val_data, col_name='val_interactions')
test_grouped = preprocessor.group_by_users(test_data, col_name='test_interactions')

data_grouped = train_grouped.merge(val_grouped, on='user_id').merge(test_grouped, on='user_id')

print(f"Number of users: {len(data_grouped)}")
print(f"Sample user data:")
print(data_grouped.head())

Number of users: 7415
Sample user data:
   user_id                                 train_interactions  \
0       11  [(8369, 95399246, -5.233), (7850, 95411006, -3...   
1       13  [(11618, 98623953, -2.672), (11209, 98624852, ...   
2       18  [(15846, 105087839, -3.838), (6457, 105089789,...   
3       25  [(24526, 105300569, -3.779), (1206, 105347095,...   
4       32  [(18225, 97610336, -3.804), (26586, 97898345, ...   

                                    val_interactions  \
0  [(8482, 110164749, -4.965), (3958, 110185189, ...   
1  [(30382, 111213199, 0.0), (17344, 111242301, -...   
2  [(2162, 109690171, -4.252), (24128, 109704924,...   
3  [(10133, 111157530, -4.484), (2137, 111175402,...   
4  [(29601, 110120451, -4.079), (18737, 110125294...   

                                   test_interactions  
0  [(18118, 112022725, -4.692), (29476, 112032227...  
1  [(8553, 111400827, -4.888), (10176, 111404767,...  
2  [(5461, 111584788, -3.207), (25542, 111614896,...  
3  [(22248, 

In [ ]:
all_train_grouped = preprocessor.group_by_users(train_data, col_name='train_interactions')

user_sequences = []
for _, row in all_train_grouped.iterrows():
    user_id = row['user_id']
    interactions = row['train_interactions']
    sequence = [item_id for item_id, _, _ in interactions]
    if len(sequence) >= 2:
        user_sequences.append((user_id, sequence))

print(f"Number of training sequences: {len(user_sequences)}")
print(f"Number of unique users: {len(set(u for u, _ in user_sequences))}")
print(f"Average sequence length: {np.mean([len(s) for _, s in user_sequences]):.2f}")
print(f"Max sequence length: {max([len(s) for _, s in user_sequences])}")
print(f"Min sequence length: {min([len(s) for _, s in user_sequences])}")

Number of training sequences: 7329
Number of unique users: 7329
Average sequence length: 122.86
Max sequence length: 2603
Min sequence length: 2


## 5. Initialize RQ-VAE Recommender

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f"Using device: {device}")

unique_users = train_data['user_id'].nunique()
print(f"Number of unique users: {unique_users:,}")

recommender = RQVAERecommender(
    item_embedding_dim=item_embeddings.shape[1],
    rqvae_hidden_dims=config['model']['rqvae']['hidden_dims'],
    num_quantizers=config['model']['rqvae']['num_quantizers'],
    codebook_size=config['model']['rqvae']['codebook_size'],
    commitment_cost=config['model']['rqvae']['commitment_cost'],
    ema_decay=config['model']['rqvae'].get('ema_decay', 0.99),
    use_gumbel=True,  # Gumbel-Softmax
    transformer_d_model=config['model']['decoder']['d_model'],
    transformer_nhead=config['model']['decoder']['nhead'],
    transformer_num_encoder_layers=3,
    transformer_num_decoder_layers=3,
    transformer_dim_feedforward=config['model']['decoder']['dim_feedforward'],
    transformer_dropout=config['model']['decoder']['dropout'],
    max_seq_length=config['model']['decoder']['max_seq_length'],
    num_users=unique_users + 100,
    device=device
)

Using device: mps
Number of unique users: 7,415

🚀 Model initialized with IMPROVED architecture!

Key improvements from official RQ-VAE-Recommender:
  ✓ Gumbel-Softmax in RQ-VAE (better gradient flow)
  ✓ Encoder-Decoder Transformer (vs decoder-only)
  ✓ User ID embeddings (personalization)
  ✓ RMSNorm (faster & more stable than LayerNorm)
  ✓ Separate input projections for encoder/decoder

RQ-VAE parameters: 834,348
Transformer parameters: 8,112,896


## 6. Train RQ-VAE (Stage 1)

Train the RQ-VAE to learn semantic IDs for items

In [ ]:
recommender.train_rqvae(
    item_embeddings=item_embeddings_tensor,
    item_ids=item_ids_array,
    epochs=config['train']['rqvae']['epochs'],
    batch_size=config['train']['rqvae']['batch_size'],
    learning_rate=config['train']['rqvae']['learning_rate']
)

print("\nRQ-VAE training completed!")

INFO:tecd_retail_recsys.models.rqvae_recommender:Training RQ-VAE on 30354 items...
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 1/60 - Loss: 2.1206, Recon: 1.0163, VQ: 1.1043, Temp: 1.000
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 10/60 - Loss: 1.2579, Recon: 1.0012, VQ: 0.2567, Temp: 0.925
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 20/60 - Loss: 0.9109, Recon: 0.5674, VQ: 0.3435, Temp: 0.842
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 30/60 - Loss: 0.9418, Recon: 0.4011, VQ: 0.5408, Temp: 0.758
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 40/60 - Loss: 0.9848, Recon: 0.3796, VQ: 0.6052, Temp: 0.675
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 50/60 - Loss: 1.0056, Recon: 0.3732, VQ: 0.6324, Temp: 0.592
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 60/60 - Loss: 1.0111, Recon: 0.3719, VQ: 0.6392, Temp: 0.508
INFO:tecd_retail_recsys.models.rqvae_recommender:Creating item to semantic ID mapping...
INFO:tecd_retail_


RQ-VAE training completed!


## 7. Analyze Semantic IDs

In [ ]:
print(f"Total items: {len(recommender.item_to_semantic_id)}")
print(f"Unique semantic IDs: {len(recommender.semantic_id_to_items)}")

collision_counts = [len(items) for items in recommender.semantic_id_to_items.values()]
collision_rate = (len([c for c in collision_counts if c > 1]) / len(collision_counts)) * 100

print(f"\nCollision statistics:")
print(f"  Collision rate: {collision_rate:.2f}%")
print(f"  Max items per semantic ID: {max(collision_counts)}")
print(f"  Average items per semantic ID: {np.mean(collision_counts):.2f}")

print(f"\nExample semantic IDs:")
for i, (item_id, sem_id) in enumerate(list(recommender.item_to_semantic_id.items())[:5]):
    print(f"  Item {item_id} -> Semantic ID: {sem_id}")

Total items: 30354
Unique semantic IDs: 21849

Collision statistics:
  Collision rate: 20.59%
  Max items per semantic ID: 24
  Average items per semantic ID: 1.39

Example semantic IDs:
  Item 20079 -> Semantic ID: (194, 135, 84, 145)
  Item 23175 -> Semantic ID: (194, 91, 233, 204)
  Item 2870 -> Semantic ID: (194, 130, 110, 241)
  Item 18639 -> Semantic ID: (194, 92, 110, 68)
  Item 14260 -> Semantic ID: (194, 166, 13, 181)


## 8. Train Transformer Decoder (Stage 2)

Train the transformer to predict next items using semantic IDs

In [ ]:

recommender.train_transformer(
    user_sequences=user_sequences,
    epochs=config['train']['decoder']['epochs'],
    batch_size=config['train']['decoder']['batch_size'],
    learning_rate=config['train']['decoder']['learning_rate'],
    warmup_epochs=config['train']['decoder'].get('warmup_epochs', 5),
)


INFO:tecd_retail_recsys.models.rqvae_recommender:Training Transformer on 7329 sequences...
INFO:tecd_retail_recsys.models.rqvae_recommender:Found 7329 unique users
INFO:tecd_retail_recsys.models.rqvae_recommender:Prepared 7329 semantic sequences
Epoch 1/10: 100%|██████████| 58/58 [02:20<00:00,  2.43s/it, loss=4.4221, lr=0.000025]
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 1/10 - Loss: 4.6767, LR: 0.000025
Epoch 2/10: 100%|██████████| 58/58 [02:19<00:00,  2.40s/it, loss=4.1040, lr=0.000050]
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 2/10 - Loss: 4.2868, LR: 0.000050
Epoch 3/10: 100%|██████████| 58/58 [02:29<00:00,  2.58s/it, loss=4.1544, lr=0.000075]
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 3/10 - Loss: 4.1875, LR: 0.000075
Epoch 4/10: 100%|██████████| 58/58 [02:17<00:00,  2.38s/it, loss=3.9875, lr=0.000100]
INFO:tecd_retail_recsys.models.rqvae_recommender:Epoch 4/10 - Loss: 4.1503, LR: 0.000100
Epoch 5/10: 100%|██████████| 58/58 [02:11<00:00,  2.27


✅ Transformer training completed with personalization!


## 9. Generate Recommendations

In [ ]:
print("Generating personalized recommendations for validation set...")
print(f"Total unique semantic IDs: {len(recommender.semantic_id_to_items)}")
print(f"Codebook size: {recommender.codebook_size}, Num quantizers: {recommender.num_quantizers}")

top_k = config['train']['top_k']
temperature = config['train']['temperature']
inference_batch_size = 256

user_data_all = [
    (row['user_id'], [item_id for item_id, _, _ in row['train_interactions']])
    for _, row in data_grouped.iterrows()
]
user_ids = data_grouped['user_id'].tolist()

all_recs: list = []
for start in tqdm(range(0, len(user_data_all), inference_batch_size), desc="Generating recs"):
    batch_data = user_data_all[start : start + inference_batch_size]
    batch_recs = recommender.recommend_batch(
        user_data=batch_data,
        top_k=top_k,
        temperature=temperature,
    )
    all_recs.extend(batch_recs)

recommendations = [
    {'user_id': uid, 'recommendations': recs}
    for uid, recs in zip(user_ids, all_recs)
]
recommendations_df = pd.DataFrame(recommendations)

print(f"\n✅ Generated personalized recommendations for {len(recommendations_df)} users")
non_empty = sum(1 for r in all_recs if len(r) > 0)
print(f"Users with non-empty recommendations: {non_empty}/{len(user_ids)}")
if non_empty > 0:
    avg_recs = np.mean([len(r) for r in all_recs if len(r) > 0])
    print(f"Average recommendations per user (non-empty): {avg_recs:.2f}")

Generating personalized recommendations for validation set...
Total unique semantic IDs: 21849
Codebook size: 256, Num quantizers: 4


Generating recs: 100%|██████████| 29/29 [12:14<00:00, 25.33s/it]


✅ Generated personalized recommendations for 7415 users
Users with non-empty recommendations: 7415/7415
Average recommendations per user (non-empty): 100.00


In [26]:
recommendations_df

,user_id,recommendations
0,11,"[(7506, -11.474365815520287), (6937, -11.64135..."
1,13,"[(7506, -11.510908395051956), (6937, -11.64111..."
2,18,"[(12508, -11.417634569108486), (7506, -11.6292..."
3,25,"[(12508, -11.301139004528522), (7506, -11.6327..."
4,32,"[(12508, -11.402410484850407), (7506, -11.6262..."
...,...,...
7410,59459,"[(7506, -11.379678972065449), (6937, -11.52539..."
7411,59476,"[(12508, -11.333178281784058), (7506, -11.6321..."
7412,59483,"[(12508, -11.296572595834732), (7506, -11.6143..."
7413,59486,"[(12508, -11.132098473608494), (7506, -11.1949..."


In [ ]:
print("Sample recommendations:")
for i in range(min(3, len(recommendations_df))):
    user_id = recommendations_df.iloc[i]['user_id']
    recs = recommendations_df.iloc[i]['recommendations']
    print(f"\nUser {user_id}:")
    print(f"  Top 10 recommendations: {[item for item, score in recs[:10]]}")
    print(f"  Scores: {[f'{score:.4f}' for item, score in recs[:10]]}")

Sample recommendations:

User 11:
  Top 10 recommendations: [7506, 6937, 9969, 6143, 24286, 30230, 13219, 12508, 13201, 18194]
  Scores: ['-11.4744', '-11.6414', '-11.6414', '-11.7858', '-11.7858', '-11.7858', '-11.7858', '-11.8091', '-11.8643', '-11.8643']

User 13:
  Top 10 recommendations: [7506, 6937, 9969, 6143, 24286, 30230, 13219, 12508, 13201, 18194]
  Scores: ['-11.5109', '-11.6411', '-11.6411', '-11.7816', '-11.7816', '-11.7816', '-11.7816', '-11.7870', '-11.8843', '-11.8843']

User 18:
  Top 10 recommendations: [12508, 7506, 13143, 6937, 9969, 6143, 24286, 30230, 13219, 14710]
  Scores: ['-11.4176', '-11.6292', '-11.7319', '-11.9662', '-11.9662', '-11.9821', '-11.9821', '-11.9821', '-11.9821', '-12.0550']


## 10. Evaluate on Validation Set

In [ ]:
eval_df = data_grouped.merge(recommendations_df, on='user_id')

print(f"Evaluation data shape: {eval_df.shape}")
print(f"Columns: {list(eval_df.columns)}")

Evaluation data shape: (7415, 5)
Columns: ['user_id', 'train_interactions', 'val_interactions', 'test_interactions', 'recommendations']


In [ ]:
print("\nCalculating metrics...\n")

metrics = calculate_metrics(
    df=eval_df,
    model_preds='recommendations',
    gt_col='val_interactions',
    train_col='train_interactions',
    verbose=True
)

print("\n" + "="*50)
print("VALIDATION METRICS")
print("="*50)
for metric_name, value in metrics.items():
    print(f"{metric_name:20s}: {value:.4f}")


Calculating metrics...

[Metrics debug] resolved gt_col='val_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (227843, 3) ratings_pred shape: (741500, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=11 gt_count=22 pred_count=100 overlap=1
  user_id=13 gt_count=5 pred_count=100 overlap=0
    [ID spaces] gt sample=[9218, 16506, 17344, 27646, 30382] range=[9218, 30382] | rec sample=[670, 722, 1548, 1792, 2210] range=[670, 30230]
  user_id=18 gt_count=47 pred_count=100 overlap=2

At k=10:
  MAP@10       = 0.0025
  NDCG@10      = 0.0104
  Precision@10 = 0.0078
  Recall@10    = 0.0030

At k=100:
  MAP@100       = 0.0014
  NDCG@100      = 0.0128
  Precision@100 = 0.0042
  Recall@100    = 0.0130

Other Metrics:
  MRR                 = 0.0235
  Catalog Coverage    = 0.0096
  Diversity     = 0.6575  [0=same recs for all, 1=unique recs]
  Novelty             =

## 11. Evaluate on Test Set

In [ ]:
print("\nCalculating test metrics...\n")

test_metrics = calculate_metrics(
    df=eval_df,
    model_preds='recommendations',
    gt_col='test_interactions',
    train_col='train_interactions',
    verbose=True
)

print("\n" + "="*50)
print("TEST METRICS")
print("="*50)
for metric_name, value in test_metrics.items():
    print(f"{metric_name:20s}: {value:.4f}")


Calculating test metrics...

[Metrics debug] resolved gt_col='test_interactions' item_id_index=0
[Metrics debug] ratings_true shape: (222882, 3) ratings_pred shape: (741500, 3)
  ratings_true dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  ratings_pred dtypes: {'user_id': dtype('int64'), 'item_id': dtype('int64')}
  user_id=11 gt_count=9 pred_count=100 overlap=1
  user_id=13 gt_count=56 pred_count=100 overlap=4
  user_id=18 gt_count=43 pred_count=100 overlap=3

At k=10:
  MAP@10       = 0.0137
  NDCG@10      = 0.0479
  Precision@10 = 0.0261
  Recall@10    = 0.0083

At k=100:
  MAP@100       = 0.0062
  NDCG@100      = 0.0464
  Precision@100 = 0.0139
  Recall@100    = 0.0436

Other Metrics:
  MRR                 = 0.0627
  Catalog Coverage    = 0.1510
  Diversity     = 0.9782  [0=same recs for all, 1=unique recs]
  Novelty             = 0.9651
  Serendipity         = -14.0179

TEST METRICS
MAP@10              : 0.0137
NDCG@10             : 0.0479
Precision@10        : 0

## 12. Save Model

In [ ]:
os.makedirs(config['info']['model_dir'], exist_ok=True)
model_path = os.path.join(config['info']['model_dir'], config['info']['checkpoint_name'])
recommender.save(model_path)
print(f"Model saved to {model_path}")

INFO:tecd_retail_recsys.models.rqvae_recommender:Model saved to ./models/rqvae/rqvae_model.pt


Model saved to ./models/rqvae/rqvae_model.pt
